## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Spring 2026

# Final Project :: Checkpoint 2


On my honor, I declare the following resources:
1. Collaborators:
- None

2. Web Sources:
- None

3. AI Tools:
- Gemini: deep research on the topic. 
- Perplexity: also deep research on the topic, but it doesn't work. apprently they nerf the deep research feature. 
- Claude Code (Sonnet 4.6): propose EDA plan, feasibility plan, and implement it. 
- GPT-5.4: create initial draft of the notebook. However it was vague and quite useless, so i delete most of it and rework with claude.
- Claude (Opus 4.6): formatting and creating the text with my prompt. 


# (A) Project Scope

## 1. Dataset Scope

This project is fully centered on the **SatHealth** dataset, a public multimodal environmental health resource designed to study relationships between localized environmental exposures and population health outcomes [1]. The dataset is geographically anchored in **Ohio** and spans **2016-2022**, with an architecture meant to scale beyond a single state. Its core analytical challenge is that it combines heterogeneous, high-dimensional, and spatially aligned modalities rather than a single flat table.

SatHealth aligns data across multiple geographic resolutions, including **Metropolitan Statistical Areas (MSAs)**, **Core Based Statistical Areas (CBSAs)**, **Census Tracts**, and **ZIP Code Tabulation Areas (ZCTAs)** [1]. Environmental factors are collected through **Google Earth Engine**, while disease prevalence is derived from the **MarketScan Commercial Claims and Encounters (CCAE)** database. Patient residencies are mapped to MSAs, and yearly prevalence for each ICD-10 code is estimated as the proportion of patients in a region who carry that diagnosis [1].

The working feature space is intentionally broad because the project asks data-mining questions that need environmental, sociodemographic, visual, and clinical context simultaneously.

| Modality Category | Data Structure | Temporal Resolution | Variable Statistics and Description |
| --- | --- | --- | --- |
| **Social Determinants of Health (SDoH)** | Tabular (spatial) | Yearly (2019 baseline) | 8 variables summarized by the Social Deprivation Index (SDI), reflecting poverty, education access, and healthcare availability [1]. |
| **Environmental Data (Land Cover)** | Tabular (spatial) | Static / time-independent | 9 variables describing land-cover fractions such as urban area, forest, and water from Copernicus land-cover products [1]. |
| **Environmental Data (Dynamic)** | Time series | Monthly (2016-2022) | 28 climate variables, 4 air-quality metrics, and 4 greenery indices collected continuously over 7 years [1]. |
| **Satellite Imagery** | High-resolution image | Static | 432,918 images at 1280 x 1280 pixels (about 500 m width), embedded into 300-dimensional vectors using RGB channels, 9 remote-sensing indices, and pixel summary statistics including mean, median, standard deviation, and 20-bin histograms [1]. |
| **Medical Records (Disease Prevalence)** | Time series | Yearly (2016-2022) | Prevalence of 1,377 ICD-10 codes derived from claims for more than 2.1 million patients [1]. |

## 2. Course-Technique Scope

Two parts of the analysis are directly tied to course material:

- **Frequent Itemset Mining (FIM) and Association Rule Generation**: used to discover cross-sectional environment-disease co-occurrence patterns that simple pairwise statistics miss.
- **Anomaly Detection / Multivariate Outlier Analysis**: used to detect localized regions whose disease profiles are unusually severe relative to their environmental context.

These course techniques are appropriate because the SatHealth tables are not well described by a single linear relationship. Disease prevalence is long-tailed, regional burden varies across urban and rural settings, and environmental variables are strongly multivariate rather than independent [1][11].

## 3. External-Technique Scope

The project also includes one explicit **external technique**: **Sequential Pattern Mining (SPM)** using **PrefixSpan** [3][4]. This extension is retained because frequent itemset mining and anomaly detection are cross-sectional, while environmental exposures and disease categories both evolve over time.

In the local repo, however, the temporal scope is narrower than the full paper-scale story. The executable data alignment yields only **14 disease-covered CBSAs** and **92 CBSA-year rows** after joining yearly disease prevalence with annualized environmental data. That is enough for a constrained sequence-construction demo, but not enough to justify strong population-scale trajectory claims without qualification.

## 4. Local Scope Constraints

The available files narrow the executable scope relative to the broader SatHealth paper.

First, the environmental CBSA tables cover **47 CBSAs**, but the ICD prevalence files in this repo cover only **14 CBSAs** with disease data. After alignment, the main executable table contains **92 CBSA-year observations**, and the disease-covered CBSA histories are only **6-7 years** long. This still supports checkpoint-level feasibility testing for RQ1 and RQ2, but it makes very low support thresholds and strong sequential claims inappropriate.

Second, CBSA-level **SDI** is not present in the release. County, CT, and ZCTA SDI tables exist, but using SDI in the primary CBSA disease workflow would require an approximate crosswalk rather than a direct merge. For this checkpoint, SDI is therefore treated as descriptive context rather than a first-pass CBSA antecedent.


# (B) Research Question Definition

## 1. Written RQ Section

**RQ1.** What are the most highly supported and confident frequent itemsets linking severe environmental deprivation (e.g., top-quartile air pollution, low greenery, high urban land cover) with the simultaneous co-occurrence of multiple chronic disease categories (e.g., metabolic and circulatory disorders) across disease-covered Ohio CBSAs?

**RQ2.** How can multivariate, unsupervised outlier detection algorithms identify severe spatial anomalies in regional disease prevalence, and do these anomalous CBSA-year observations align with extreme, localized deviations in satellite-derived environmental factors?

**RQ3.** What are the dominant sequential trajectories of disease progression when mediated by prolonged, multi-year exposure to deteriorating environmental conditions over the 2016–2022 timeline, and can a constrained PrefixSpan configuration recover any short recurring disease-environment sequences from the aligned yearly data?

## 2. RQ-by-RQ Detail

### RQ1: Uncovering Environment-Disease Co-occurrences via Frequent Itemset Mining

The first research question addresses the static, cross-sectional correlations surfaced during the exploratory data analysis. While simple pairwise statistics such as odds ratios can establish basic associations, they fail to uncover complex, multi-variable interactions — for instance, the precise intersection of high ozone levels, elevated urban land cover, and low greenery that consistently co-occurs with high diabetes and circulatory disease prevalence [5].

- **Technique used**: Frequent Itemset Mining (FIM) and subsequent Association Rule Generation.
- **Course or external**: **In-class / course technique**.
- **Data mining task type**: frequent itemset discovery and rule extraction from transactionalized environmental and disease indicators.
- **Relevant algorithms**: **Apriori** as the foundational, breadth-first baseline and **FP-Growth** as the primary execution engine [5]. The Apriori algorithm operates on the downward closure property — any subset of a frequent itemset must also be frequent — and iteratively generates candidate itemsets of increasing length. However, as confirmed in the feasibility pre-trials, Apriori encounters severe memory pressure at low support thresholds due to combinatorial candidate explosion. FP-Growth circumvents this bottleneck entirely by compressing the transactional database into a condensed Frequent Pattern Tree (FP-Tree) and applying recursive pattern-fragment growth, efficiently mining deep, long-tail combinations that Apriori fails to compute in a reasonable time frame [5].
- **Why these methods fit**: the annualized CBSA-year table can be discretized into interpretable environmental quartile bins, while ICD level-1 prevalence can be converted into high-prevalence disease events. This preserves interpretability and keeps the first executable transaction matrix small enough for a checkpoint dry run. Disease prevalence in SatHealth is long-tailed, regional burden varies across urban and rural settings, and environmental variables are strongly multivariate rather than independent [1][11], which makes combinatorial pattern discovery more informative than simple pairwise analysis.
- **Evaluation criteria**: the analysis will prioritize **Lift** alongside support and confidence. Lift evaluates the ratio of the observed support to what would be expected if antecedent and consequent were independent; a value significantly greater than 1.0 indicates that the environmental condition actively increases the likelihood of the disease beyond its baseline prevalence, moving the analysis from statistical coincidence toward actionable insight [5]. Runtime comparisons between Apriori and FP-Growth are also part of the feasibility argument. Because the aligned table contains 92 rows, the minimum meaningful support floor in this notebook is **3 transactions** rather than an arbitrarily tiny fraction.

### RQ2: Spatiotemporal Biosurveillance via Ensemble Anomaly Detection

The second research question transitions from global pattern recognition to localized outlier identification, driven by the public health imperative for rapid, geographically precise biosurveillance. The objective is to computationally isolate specific CBSA-year observations whose disease profiles drastically deviate from the broader Ohio baseline, which is critical for identifying potential unrecorded localized hazards [7].

- **Technique used**: unsupervised multivariate anomaly detection.
- **Course or external**: **In-class / course technique**.
- **Data mining task type**: outlier detection over region-level environmental and disease feature matrices.
- **Relevant algorithms**: **Isolation Forest (iForest)** as the primary detector and **Local Outlier Factor (LOF)** as the density-based consensus validator [6][7]. Traditional anomaly detection models such as Gaussian Z-score methods assume normality, which the SatHealth disease prevalence data violates due to its power-law, long-tail distribution. Isolation Forest bypasses this limitation because it does not rely on density or distance scaling; instead, it operates on the principle that anomalies are "few and different" [6]. The algorithm generates an ensemble of random decision trees, where anomalous observations require significantly fewer random splits to be isolated into terminal nodes. LOF complements iForest by checking whether the same flagged points also appear locally sparse relative to their nearest neighbors [7].
- **Why these methods fit**: the cleaned joint feature matrix is multivariate, non-Gaussian, and small enough that detector overlap can be inspected directly. Pre-trial testing revealed high sensitivity to the contamination hyperparameter — arbitrary settings generated excessive false positives, flagging standard urban healthcare variations as severe anomalies. This confirmed the necessity of a dual-algorithm consensus approach to mitigate alarm fatigue [6][7].
- **Evaluation criteria**: anomaly score rankings, detector overlap between iForest and LOF, contamination sensitivity across multiple low settings, and qualitative plausibility of flagged CBSA-years.

### RQ3: Mapping Chronological Disease Trajectories via Sequential Pattern Mining

The third research question addresses a fundamental limitation of the previous two methods: both Frequent Itemset Mining and standard Anomaly Detection treat data cross-sectionally and are blind to the passage of time. In clinical reality, diseases operate sequentially — prolonged exposure to deteriorating environmental conditions may trigger cascading health outcomes over multiple years [8][10]. Traditional static correlation analysis lacks the architectural mechanisms to capture this chronology.

- **Technique used**: Sequential Pattern Mining (SPM).
- **Course or external**: **External technique**.
- **Data mining task type**: discovery of short ordered disease-environment sequences over yearly CBSA histories.
- **Relevant algorithms**: **PrefixSpan** (Prefix-projected Sequential Pattern mining) as the primary algorithm [3][4]. Early sequential algorithms such as GSP and SPADE rely on candidate generation and vertical database intersections that frequently lead to memory exhaustion on dense datasets [9]. PrefixSpan circumvents this by constructing projected databases for each frequent prefix, recursively mining only the relevant postfixes. This projection-based architecture is fundamentally more memory-efficient than vertical-format algorithms, making it the preferred choice for the variable-length sequences in SatHealth [3][8].
- **Why these methods fit**: each disease-covered CBSA becomes a yearly sequence of environmental regime tokens plus high-prevalence disease tokens. PrefixSpan mines ordered subsequences without generating every possible candidate. In this checkpoint it is used only as a **feasibility demo**, because the aligned data yields just 14 sequences of length 6–7. Strict length constraints (`maxlen`) are enforced to prevent excessive recursive projection [4].
- **Evaluation criteria**: minimum support count, pattern length, whether any patterns contain both environmental and disease tokens, and runtime. A negative result is still informative here because the main goal is to test constructability rather than to claim a strong clinical pathway discovery.

## 3. RQ-to-Method Mapping Table

| RQ | Analytical Objective | Data Mining Task Type | Relevant Algorithm(s) | Course vs External | Evaluation Criteria |
| --- | --- | --- | --- | --- | --- |
| **RQ1: Environment-Disease Associations** | Discover non-linear, multi-variable co-occurrences of environmental hazards and chronic disease clusters across disease-covered Ohio CBSAs. | Frequent Itemset Mining and association-rule generation | Apriori (baseline), FP-Growth (primary) | Course / in-class | Support count, confidence, lift, runtime |
| **RQ2: CBSA-Year Biosurveillance** | Identify CBSA-year observations whose disease profiles are statistically anomalous relative to their surrounding environmental baselines. | Anomaly detection / multivariate outlier analysis | Isolation Forest (primary), LOF (consensus validator) | Course / in-class | Score rankings, detector overlap, contamination sensitivity |
| **RQ3: Temporal Disease Trajectories** | Map chronological, ordered disease-environment progression patterns over the 2016–2022 timeline and test whether short recurring sequences can be recovered from the aligned yearly data. | Sequential Pattern Mining | PrefixSpan (primary feasibility demo) | External | Support count, pattern length, mixed env-disease content, runtime |

# (C) Motivation and Feasibility

## 1. Motivation

The motivation for this project is that environmentally mediated disease burden is unlikely to be captured by simple one-variable-at-a-time analysis. SatHealth is valuable because it bridges environmental sensing and clinical outcomes, which makes it a realistic setting for discovering actionable public-health patterns rather than only fitting narrow predefined models [1].

Each research question captures a different part of that motivation. **RQ1** targets hidden combinatorial risk patterns that would be missed by pairwise summaries alone. **RQ2** targets automated biosurveillance: the ability to flag region-years whose disease mix looks unusually severe relative to their surroundings. **RQ3** targets temporal constructability: whether even a constrained version of ordered disease-environment mining is possible once the data are aligned to yearly histories.

## 2. Non-Triviality

This project remains non-trivial for four reasons. First, the dataset is **multimodal**, combining tabular, time-series, image-derived, sociodemographic, and medical prevalence data. Second, it is **high-dimensional**, especially on the disease side. Third, its structure is **non-linear and spatially heterogeneous**, with meaningful urban-rural differences and localized environmental phenotypes. Fourth, its modalities are **temporally misaligned by design**, because environmental variables are often monthly while disease prevalence is yearly [1].

Those properties make simple one-model baselines insufficient for the overall project goal. Traditional regression still has value later, but it is not the right first tool for high-order co-occurrence discovery, unsupervised anomaly screening, or ordered subsequence mining.

## 3. Feasibility

The feasibility claims in this checkpoint are based on the **local dataset and executable notebook cells below**, not on off-notebook pre-trials.

A direct audit of the repo shows that the CBSA environmental tables cover **47 CBSAs**, while the ICD prevalence tables cover **14 CBSAs**. After annualizing monthly environmental variables and merging them with ICD level-1 disease prevalence, the executable joint table contains **92 CBSA-year rows**. That is enough to support:

- a first-pass transaction matrix for frequent itemset mining,
- anomaly detection on a modest multivariate feature matrix, and
- a tightly bounded sequence-construction demo over 14 yearly CBSA histories.

The same audit also surfaces the main practical constraints. `seaborn`, `mlxtend`, and `prefixspan` are missing in the current environment and therefore need explicit bootstrap code. CBSA SDI is not directly available. The joint table contains material missingness in greenery and NO2, while some environmental variables such as PM2.5 are effectively near-constant in this extract. Finally, the climate block is highly collinear, which means correlation pruning is necessary before anomaly modeling.

| Algorithmic Paradigm | Local Evidence Used in This Notebook | Main Risk | Mitigation in This Checkpoint |
| --- | --- | --- | --- |
| **Frequent Itemset Mining** | 92 aligned CBSA-year rows and executable environmental discretization into bins | Support set too low to be meaningful or candidate space grows too quickly | Keep the support floor at **3 transactions**, use ICD level-1 disease categories, and compare Apriori with FP-Growth on the same transaction matrix |
| **Anomaly Detection** | Cleaned joint matrix built from annualized CBSA environment plus ICD level-1 prevalence | Detector output can change under different contamination settings | Run iForest and LOF at two low contamination settings and inspect overlap rather than trusting a single score |
| **Sequential Mining** | 14 yearly CBSA histories of length 6-7 constructed from the aligned table | Very small sample of sequences can make substantive pattern claims fragile | Treat PrefixSpan as a feasibility demo only, cap `maxlen` at 3, and keep minimum support at 3 sequences |

## 4. Risk

The main risks are computational and methodological rather than conceptual.

- **Support dilution in frequent itemset mining**: with only 92 CBSA-year rows, thresholds below 3 supporting transactions are not meaningful. The mitigation is to use count-based support floors and keep the disease granularity at ICD level 1 for the first pass.
- **Parameter sensitivity in anomaly detection**: arbitrary contamination settings can inflate false positives. The mitigation is to compare multiple low contamination settings and emphasize detector overlap.
- **Sequence sparsity**: 14 sequences are enough for feasibility testing but not for strong trajectory claims. The mitigation is to frame RQ3 as proof-of-constructability and retain negative results if the support is too weak.
- **Cross-geography mismatch**: SDI exists below the CBSA level, not at the CBSA level used for disease prevalence. The mitigation is to exclude SDI from the primary executable workflow in this checkpoint.


# (D) Methodological Planning

This section turns the research questions into an execution plan that clearly separates course methods from the external method, records the main constraints, and ties each claim to an executable notebook step.

## 1. Course Algorithms

For **RQ1**, the course baseline is **Apriori**, while **FP-Growth** is the main execution engine. The notebook uses an annualized CBSA-year table, discretizes a small set of stable environmental variables into quartile-based items, and converts ICD level-1 prevalence into high-prevalence disease events. Rules are retained only when the antecedent is environmental and the support floor is at least **3 of 92 transactions**.

For **RQ2**, the core course algorithm is **Isolation Forest**. It operates on a cleaned CBSA-year feature matrix created after median imputation, missing-indicator creation, near-zero variance removal, and high-correlation pruning. **LOF** is retained as the second detector because consensus is more informative here than a single anomaly score.

## 2. External Algorithm

For **RQ3**, the external method is **PrefixSpan** implemented through the `prefixspan` Python package [3][4]. The representation is intentionally coarse: each disease-covered CBSA becomes a yearly sequence of environmental regime tokens plus high-prevalence disease tokens. This is not intended to be the final project's definitive temporal model; it is a checkpoint feasibility probe.

## 3. Method and Metric Plan

| Algorithmic Phase | Primary Algorithm | Course vs External | Hyperparameter / Execution Constraints | Evaluation Metrics | Baseline / Comparator |
| --- | --- | --- | --- | --- | --- |
| **Frequent Itemset Mining** | FP-Growth | Course / in-class | Run at `min_support = 0.05` and `min_support = 3 / 92`; keep only rules with environmental antecedents; use ICD level-1 disease events and 5-8 discretized environmental variables | Support count, confidence, lift, runtime | **Apriori** on the same transaction matrix |
| **Anomaly Detection** | Isolation Forest | Course / in-class | Set `n_estimators = 100`; evaluate contamination at `0.03` and `0.05`; standardize the pruned numeric matrix and compare to LOF | Score rankings, overlap count, contamination sensitivity | **LOF** as the density-based comparator |
| **Sequential Pattern Mining** | PrefixSpan | External | Use yearly CBSA sequences with `minlen = 2`, `maxlen = 3`, and minimum support of **3 sequences**; keep the result interpretation explicitly exploratory | Support count, pattern length, mixed env-disease content, runtime | No executed checkpoint baseline; SPADE remains a literature reference only |

## 4. Evaluation Logic by Research Question

- **RQ1 evaluation logic**: strong rules should have non-trivial support, high confidence, and lift above 1. Runtime also matters because Apriori versus FP-Growth is part of the feasibility argument.
- **RQ2 evaluation logic**: useful anomaly detectors should surface a short list of plausible CBSA-year outliers, and that short list should not change wildly across small contamination adjustments.
- **RQ3 evaluation logic**: a successful checkpoint outcome is either a small set of short recurring mixed patterns or a clear negative result that shows support is too weak for stronger claims.

## 5. Planned Baselines

- **Apriori** is the baseline for FP-Growth in RQ1.
- **LOF** is the comparison detector for iForest in RQ2.
- **SPADE** is not executed in this checkpoint; it remains only a conceptual reference point from the literature for why a bounded PrefixSpan demo is preferable here.

Together, these choices keep the notebook methodologically grounded while avoiding claims that the local data and environment cannot support.


# (E) Local Feasibility Checks and Initial Method Runs

The next cells provide the executable evidence for the feasibility claims above. They are ordered to make the workflow auditable from top to bottom:

1. bootstrap the Python environment and install any missing analysis packages,
2. audit local table coverage and the true size of the aligned CBSA-year data,
3. run targeted EDA needed for method feasibility,
4. construct the cleaned joint analytic table,
5. execute initial dry runs for RQ1, RQ2, and RQ3.

If the bootstrap cell cannot install missing packages, the rest of the method cells should be treated as blocked rather than as evidence against the methods themselves.


In [ ]:
# Environment bootstrap: imports, package checks, and notebook-safe matplotlib config.
import importlib
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

# WHY tempdir for MPLCONFIGDIR: avoids permission errors when matplotlib tries to
# write its font cache to the default home directory, which may be read-only in
# some notebook environments (e.g., shared JupyterHub, containerized kernels).
mpl_dir = Path(tempfile.mkdtemp(prefix="mplconfig_"))
os.environ["MPLCONFIGDIR"] = str(mpl_dir)

REQUIRED_MODULES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "seaborn": "seaborn",
    "mlxtend": "mlxtend",
    "prefixspan": "prefixspan",
}

missing_packages = []
for module_name, package_name in REQUIRED_MODULES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

print("MPLCONFIGDIR:", os.environ["MPLCONFIGDIR"])
print("Missing packages before bootstrap:", missing_packages if missing_packages else "None")

if missing_packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Automatic package installation failed. Re-run this cell after enabling network access or install seaborn, mlxtend, and prefixspan manually."
        ) from exc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from prefixspan import PrefixSpan
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

versions = {}
for module_name in REQUIRED_MODULES:
    module = importlib.import_module(module_name)
    versions[module_name] = getattr(module, "__version__", "unknown")

print("\nResolved package versions:")
for name, version in versions.items():
    print(f"- {name}: {version}")

DATA_DIR = Path("sathealth_dataset")
ID_COLS = ["CBSAFP", "year"]

def annualize_monthly(df, geo_col):
    """Collapse monthly rows to yearly averages per geographic unit.
    
    WHY mean aggregation: environmental variables like temperature, precipitation,
    and air quality are measured monthly but disease prevalence is reported annually.
    Averaging preserves the central tendency of each year's exposure while matching
    the temporal grain of the health outcome data.
    """
    keep = [geo_col, "year"] + [
        c
        for c in df.select_dtypes(include=[np.number]).columns
        if c not in {geo_col, "year", "month"}
    ]
    return df[keep].groupby([geo_col, "year"], as_index=False).mean()

def top_abs_correlations(df, top_n=10):
    """Return the top-N highest absolute pairwise correlations.
    
    WHY absolute value: we care about redundancy regardless of sign — two features
    correlated at -0.99 are just as redundant as +0.99 for downstream modeling.
    """
    corr = df.corr(numeric_only=True).abs()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    pairs = corr.where(mask).stack().sort_values(ascending=False).reset_index()
    pairs.columns = ["feature_1", "feature_2", "abs_corr"]
    return pairs.head(top_n)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [2]:
# Coverage and schema audit.
cbsa_aq = pd.read_csv(DATA_DIR / "CBSA" / "airquality.csv")
cbsa_cl = pd.read_csv(DATA_DIR / "CBSA" / "climate.csv")
cbsa_gr = pd.read_csv(DATA_DIR / "CBSA" / "greenery.csv")
cbsa_lc = pd.read_csv(DATA_DIR / "CBSA" / "landcover.csv")
county_sdi = pd.read_csv(DATA_DIR / "County" / "sdi.csv")
icd1 = pd.read_csv(DATA_DIR / "icdl1_prev_ohio.csv")

annual_cbsa_aq = annualize_monthly(cbsa_aq, "CBSAFP")
annual_cbsa_cl = annualize_monthly(cbsa_cl, "CBSAFP")
annual_cbsa_gr = annualize_monthly(cbsa_gr, "CBSAFP")

joint_keys = icd1[ID_COLS].drop_duplicates().sort_values(ID_COLS).reset_index(drop=True)
years_per_cbsa = joint_keys.groupby("CBSAFP")["year"].nunique()

audit_df = pd.DataFrame(
    [
        {
            "table": "CBSA airquality",
            "rows": cbsa_aq.shape[0],
            "cols": cbsa_aq.shape[1],
            "unique_geo": cbsa_aq["CBSAFP"].nunique(),
            "year_min": int(cbsa_aq["year"].min()),
            "year_max": int(cbsa_aq["year"].max()),
        },
        {
            "table": "CBSA climate",
            "rows": cbsa_cl.shape[0],
            "cols": cbsa_cl.shape[1],
            "unique_geo": cbsa_cl["CBSAFP"].nunique(),
            "year_min": int(cbsa_cl["year"].min()),
            "year_max": int(cbsa_cl["year"].max()),
        },
        {
            "table": "CBSA greenery",
            "rows": cbsa_gr.shape[0],
            "cols": cbsa_gr.shape[1],
            "unique_geo": cbsa_gr["CBSAFP"].nunique(),
            "year_min": int(cbsa_gr["year"].min()),
            "year_max": int(cbsa_gr["year"].max()),
        },
        {
            "table": "CBSA landcover",
            "rows": cbsa_lc.shape[0],
            "cols": cbsa_lc.shape[1],
            "unique_geo": cbsa_lc["CBSAFP"].nunique(),
            "year_min": np.nan,
            "year_max": np.nan,
        },
        {
            "table": "County SDI",
            "rows": county_sdi.shape[0],
            "cols": county_sdi.shape[1],
            "unique_geo": county_sdi["COUNTYFP"].nunique(),
            "year_min": np.nan,
            "year_max": np.nan,
        },
        {
            "table": "ICD L1 prevalence",
            "rows": icd1.shape[0],
            "cols": icd1.shape[1],
            "unique_geo": icd1["CBSAFP"].nunique(),
            "year_min": int(icd1["year"].min()),
            "year_max": int(icd1["year"].max()),
        },
    ]
)

env_coverage = joint_keys.merge(annual_cbsa_aq, on=ID_COLS, how="left")
env_coverage = env_coverage.merge(annual_cbsa_cl, on=ID_COLS, how="left", suffixes=("_aq", "_cl"))
env_coverage = env_coverage.merge(annual_cbsa_gr, on=ID_COLS, how="left", suffixes=("", "_gr"))
env_coverage = env_coverage.merge(cbsa_lc, on="CBSAFP", how="left")

print("Table audit:")
display(audit_df)

print("\nKey feasibility facts:")
print(f"- Environmental CBSAs in local CBSA tables: {cbsa_aq['CBSAFP'].nunique()}")
print(f"- Disease-covered CBSAs in ICD L1 table: {icd1['CBSAFP'].nunique()}")
print(f"- Joint CBSA-year rows after alignment: {len(joint_keys)}")
print(f"- Sequence length range by disease-covered CBSA: {int(years_per_cbsa.min())} to {int(years_per_cbsa.max())} years")
print(f"- Rows with at least one missing value after annual environment merge: {int(env_coverage.isna().any(axis=1).sum())}")

print("\nYears per disease-covered CBSA summary:")
display(years_per_cbsa.describe().to_frame(name="n_years"))


Table audit:


,table,rows,cols,unique_geo,year_min,year_max
0,CBSA airquality,3948,7,47,2016.0,2022.0
1,CBSA climate,3948,31,47,2016.0,2022.0
2,CBSA greenery,3948,7,47,2016.0,2022.0
3,CBSA landcover,47,11,47,NaN,NaN
4,County SDI,88,18,88,NaN,NaN
5,ICD L1 prevalence,1969,6,14,2016.0,2022.0



Key feasibility facts:
- Environmental CBSAs in local CBSA tables: 47
- Disease-covered CBSAs in ICD L1 table: 14
- Joint CBSA-year rows after alignment: 92
- Sequence length range by disease-covered CBSA: 6 to 7 years
- Rows with at least one missing value after annual environment merge: 31

Years per disease-covered CBSA summary:


,n_years
count,14.000000
mean,6.571429
std,0.513553
min,6.000000
25%,6.000000
50%,7.000000
75%,7.000000
max,7.000000


In [ ]:
# Additional EDA needed to assess method feasibility.
joint_aq = joint_keys.merge(annual_cbsa_aq, on=ID_COLS, how="left")
joint_cl = joint_keys.merge(annual_cbsa_cl, on=ID_COLS, how="left")
joint_gr = joint_keys.merge(annual_cbsa_gr, on=ID_COLS, how="left")
joint_env = joint_keys.merge(annual_cbsa_cl, on=ID_COLS, how="left")
joint_env = joint_env.merge(annual_cbsa_aq, on=ID_COLS, how="left")
joint_env = joint_env.merge(annual_cbsa_gr, on=ID_COLS, how="left")
joint_env = joint_env.merge(cbsa_lc, on="CBSAFP", how="left")

air_missing = (joint_aq.isna().mean() * 100).sort_values(ascending=False).rename("missing_pct")
green_missing = (joint_gr.isna().mean() * 100).sort_values(ascending=False).rename("missing_pct")

pm25_col = "particulate_matter_d_less_than_25_um_surface"
pm25_stats = pd.Series(
    {
        "non_null_count": int(joint_aq[pm25_col].notna().sum()),
        "nunique": int(joint_aq[pm25_col].nunique(dropna=True)),
        "std": float(joint_aq[pm25_col].std()),
        "min": float(joint_aq[pm25_col].min()),
        "max": float(joint_aq[pm25_col].max()),
    },
    name=pm25_col,
)

climate_numeric = annual_cbsa_cl.drop(columns=ID_COLS)
climate_corr = top_abs_correlations(climate_numeric, top_n=10)

env_numeric = joint_env.drop(columns=ID_COLS)
env_missing = env_numeric.isna().mean()
env_std = env_numeric.std(numeric_only=True)
env_nunique = env_numeric.nunique(dropna=True)
# WHY these three filters for safe_env_candidates:
#   - missing <= 25%: features with more than a quarter of values missing would need
#     heavy imputation, risking artificial patterns in the discretized transactions.
#   - nunique > 10: features with fewer than 10 distinct values are effectively
#     categorical already and may not discretize into meaningful quartile bins.
#   - std > 1e-6: near-constant features carry no discriminative information and
#     would produce degenerate bins where all rows fall into the same quartile.
safe_env_candidates = [
    col
    for col in env_numeric.columns
    if env_missing.get(col, 1.0) <= 0.25 and env_nunique.get(col, 0) > 10 and env_std.get(col, 0.0) > 1e-6
]

print("Annualized air-quality missingness on disease-covered CBSA-years (%):")
display(air_missing.head(6).to_frame())

print("\nAnnualized greenery missingness on disease-covered CBSA-years (%):")
display(green_missing.head(6).to_frame())

print("\nNear-constant PM2.5 check:")
display(pm25_stats.to_frame())

print("\nTop absolute climate correlations:")
display(climate_corr)

print("\nCandidate environmental features that are safe for a first-pass discretization / anomaly model:")
print(safe_env_candidates[:15])

In [ ]:
# Joint analytic table builder: annualize, merge, impute, and prune.
icd1_wide = icd1.pivot_table(index=ID_COLS, columns="code", values="prevalence").reset_index()
DISEASE_COLUMNS = sorted(icd1["code"].unique().tolist())
# WHY fillna(0.0): a missing cell in the ICD pivot means that specific CBSA-year had
# zero observed patients with that ICD code — absence of a prevalence row is
# semantically equivalent to zero prevalence, not unknown data.
icd1_wide[DISEASE_COLUMNS] = icd1_wide[DISEASE_COLUMNS].fillna(0.0)

# WHY left joins on environment, inner join on ICD: we keep every CBSA-year that has
# disease data (the analysis unit) and attach environmental features where available;
# the inner join on ICD ensures we never model rows with no health outcome.
analysis_raw_df = joint_keys.merge(annual_cbsa_cl, on=ID_COLS, how="left")
analysis_raw_df = analysis_raw_df.merge(annual_cbsa_aq, on=ID_COLS, how="left")
analysis_raw_df = analysis_raw_df.merge(annual_cbsa_gr, on=ID_COLS, how="left")
analysis_raw_df = analysis_raw_df.merge(cbsa_lc, on="CBSAFP", how="left")
analysis_raw_df = analysis_raw_df.merge(icd1_wide, on=ID_COLS, how="inner")
analysis_raw_df = analysis_raw_df.sort_values(ID_COLS).reset_index(drop=True)

analysis_df = analysis_raw_df.copy()
# WHY drop all-null columns: columns with 100% missing values carry zero information
# and would cause errors in downstream imputation and modeling steps.
all_null_cols = [col for col in analysis_df.columns if analysis_df[col].isna().all()]
if all_null_cols:
    analysis_df = analysis_df.drop(columns=all_null_cols)

numeric_cols = [col for col in analysis_df.select_dtypes(include=[np.number]).columns if col not in ID_COLS]
# WHY 40% missing threshold: columns missing more than 40% of values would require
# heavy imputation that risks injecting more noise than signal. The 40% cutoff is a
# conservative balance — strict enough to avoid unreliable features, lenient enough
# to retain columns where most data is present.
moderate_missing_cols = [
    col for col in numeric_cols if 0 < analysis_df[col].isna().mean() <= 0.40
]
missing_indicator_cols = []
for col in moderate_missing_cols:
    # WHY missing-indicator + median imputation: the indicator column lets downstream
    # models learn whether "missingness itself" is informative (e.g., a sensor offline
    # in rural areas). Median is robust to outliers, unlike mean, and does not assume
    # normality — appropriate for environmental measurements that are often skewed.
    indicator_col = f"{col}_missing"
    analysis_df[indicator_col] = analysis_df[col].isna().astype(int)
    analysis_df[col] = analysis_df[col].fillna(analysis_df[col].median())
    missing_indicator_cols.append(indicator_col)

numeric_cols = [col for col in analysis_df.select_dtypes(include=[np.number]).columns if col not in ID_COLS]
# WHY remove near-constant columns: features with <= 1 unique value or std < 1e-8
# provide no discriminative power for any model and can cause numerical instability
# (e.g., division by zero during standardization).
near_constant_cols = [
    col for col in numeric_cols if analysis_df[col].nunique(dropna=True) <= 1 or analysis_df[col].std() < 1e-8
]
if near_constant_cols:
    analysis_df = analysis_df.drop(columns=near_constant_cols)

numeric_cols = [col for col in analysis_df.select_dtypes(include=[np.number]).columns if col not in ID_COLS]
corr = analysis_df[numeric_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
# WHY 0.98 correlation threshold: pairs above 0.98 are near-duplicates that inflate
# dimensionality without adding information. A threshold below 0.90 would be too
# aggressive and risk dropping features with meaningfully different signals; 0.98
# targets only truly redundant pairs while preserving moderately correlated features
# that may still carry independent variance.
high_corr_drop = sorted([col for col in upper.columns if any(upper[col] >= 0.98)])

analysis_model_df = analysis_df.drop(columns=high_corr_drop)
ENV_COLUMNS = [col for col in analysis_model_df.columns if col not in ID_COLS + DISEASE_COLUMNS]
MODEL_FEATURE_COLUMNS = [col for col in analysis_model_df.columns if col not in ID_COLS]

print("Joint analytic table build summary:")
print(f"- Raw aligned table shape: {analysis_raw_df.shape}")
print(f"- All-null columns dropped: {len(all_null_cols)}")
print(f"- Missing-indicator columns added: {len(missing_indicator_cols)}")
print(f"- Near-constant columns dropped: {len(near_constant_cols)}")
print(f"- High-correlation columns dropped (|r| >= 0.98): {len(high_corr_drop)}")
print(f"- Final modeling table shape: {analysis_model_df.shape}")
print(f"- Final modeling feature count (excluding IDs): {len(MODEL_FEATURE_COLUMNS)}")

if high_corr_drop:
    print("\nSample of pruned high-correlation columns:")
    print(high_corr_drop[:15])

In [ ]:
# Initial RQ1 dry run: transaction building plus Apriori / FP-Growth checks.

# WHY these 7 features: they span the three environmental modalities in the dataset
# (air quality, climate, greenery/land cover), are among the safe_env_candidates
# identified in the EDA cell, and are domain-interpretable for public-health readers.
# Keeping the feature set small (7) avoids combinatorial explosion in itemset mining.
RULE_ENV_FEATURES = [
    "ozone",
    "temperature_2m",
    "total_precipitation_sum",
    "surface_pressure",
    "leaf_area_index_high_vegetation",
    "urban-coverfraction",
    "tree-coverfraction",
]
RULE_ENV_FEATURES = [col for col in RULE_ENV_FEATURES if col in analysis_df.columns]

rule_work_df = analysis_df[ID_COLS + RULE_ENV_FEATURES + DISEASE_COLUMNS].copy()
env_bins = {}
for col in RULE_ENV_FEATURES:
    # WHY quartile (4-bin) discretization via qcut on ranks: quartiles split data into
    # four equal-frequency groups, which guarantees each bin has enough transactions to
    # meet reasonable support thresholds. Rank-based qcut avoids sensitivity to outliers
    # and skewed distributions that would produce empty bins with equal-width cuts.
    env_bins[col] = pd.qcut(
        rule_work_df[col].rank(method="first"),
        4,
        labels=["Q1", "Q2", "Q3", "Q4"],
    ).astype(str)

# WHY 75th-percentile threshold for diseases: we want transactions to flag "elevated"
# prevalence, not just any nonzero value. The 75th percentile marks the top quartile,
# creating a natural binary split that aligns with the quartile scheme used for
# environmental features and focuses association rules on clinically notable burdens.
disease_thresholds = rule_work_df[DISEASE_COLUMNS].quantile(0.75)
transactions = []
for idx, row in rule_work_df.iterrows():
    tx = []
    for col in RULE_ENV_FEATURES:
        tx.append(f"ENV:{col}={env_bins[col].iloc[idx]}")
    for code in DISEASE_COLUMNS:
        value = row[code]
        threshold = disease_thresholds[code]
        if pd.notna(value) and value >= threshold:
            tx.append(f"DX:{code}")
    transactions.append(tx)

te = TransactionEncoder()
transaction_matrix = te.fit(transactions).transform(transactions)
transaction_df = pd.DataFrame(transaction_matrix, columns=te.columns_)

# WHY filter rules to ENV-only antecedents and DX-containing consequents: the research
# question asks "which environmental conditions co-occur with disease burden?" — rules
# where diseases predict other diseases or environments predict environments are not
# actionable for that question.
def env_antecedent_only(itemset):
    return len(itemset) > 0 and all(item.startswith("ENV:") for item in itemset)

def disease_consequent_present(itemset):
    return any(item.startswith("DX:") for item in itemset)

# WHY two support values [0.05, 3/n]: 0.05 is a standard starting point in the
# association-rule literature; 3/n is the absolute floor (an itemset appearing in at
# least 3 transactions) to test how far we can lower support before the search space
# explodes. Comparing both reveals sensitivity of the rule set to the threshold.
support_values = [0.05, 3 / len(transaction_df)]
rq1_summary_rows = []
rq1_rule_tables = {}

for method_name, method_fn in [("Apriori", apriori), ("FP-Growth", fpgrowth)]:
    for support in support_values:
        started = time.perf_counter()
        frequent_itemsets = method_fn(
            transaction_df,
            min_support=support,
            use_colnames=True,
            # WHY max_len=3: limits itemsets to at most 3 items to keep rules
            # human-interpretable (e.g., "high ozone + high urban cover → elevated
            # respiratory disease"). Longer itemsets become hard to act on and are
            # more likely to overfit the small dataset.
            max_len=3,
        )
        runtime = time.perf_counter() - started

        if frequent_itemsets.empty:
            rq1_summary_rows.append(
                {
                    "method": method_name,
                    "min_support": round(support, 4),
                    "itemsets_found": 0,
                    "rules_retained": 0,
                    "runtime_sec": round(runtime, 4),
                }
            )
            continue

        # WHY confidence >= 0.6: a rule must hold in at least 60% of transactions
        # containing its antecedent to be considered reliable. This is a moderate
        # threshold — low enough to surface exploratory patterns in a small dataset,
        # high enough to filter out noise.
        rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
        rules = rules[
            rules["antecedents"].apply(env_antecedent_only)
            & rules["consequents"].apply(disease_consequent_present)
        ].sort_values(["lift", "confidence", "support"], ascending=False)

        rq1_summary_rows.append(
            {
                "method": method_name,
                "min_support": round(support, 4),
                "itemsets_found": int(len(frequent_itemsets)),
                "rules_retained": int(len(rules)),
                "runtime_sec": round(runtime, 4),
            }
        )
        rq1_rule_tables[(method_name, round(support, 4))] = rules[
            ["antecedents", "consequents", "support", "confidence", "lift"]
        ].head(10)

rq1_summary = pd.DataFrame(rq1_summary_rows)
print("RQ1 transaction audit:")
print(f"- Transaction rows: {len(transaction_df)}")
print(f"- Environmental features discretized: {len(RULE_ENV_FEATURES)}")
print(f"- Support floor from 3 transactions: {3 / len(transaction_df):.4f}")
display(rq1_summary)

for key, table in rq1_rule_tables.items():
    method_name, support = key
    print(f"\nTop retained rules for {method_name} at min_support={support}:")
    if table.empty:
        print("No retained rules at this threshold.")
    else:
        display(table)

In [ ]:
# Initial RQ2 dry run: Isolation Forest plus LOF consensus checks.
rq2_feature_df = analysis_model_df[MODEL_FEATURE_COLUMNS].copy()
# WHY StandardScaler: Isolation Forest and LOF both operate on distance / partition
# metrics that are sensitive to feature scale. Without standardization, features with
# large numeric ranges (e.g., surface pressure in Pa) would dominate the anomaly score
# over features with small ranges (e.g., prevalence proportions between 0 and 1).
rq2_scaler = StandardScaler()
rq2_matrix = rq2_scaler.fit_transform(rq2_feature_df)

rq2_summary_rows = []
rq2_detail_tables = {}
# WHY n_neighbors = min(10, n-1): LOF needs a neighborhood size to estimate local
# density. 10 is the low end of the commonly recommended range (10–20) and appropriate
# for a small dataset where larger neighborhoods would over-smooth density estimates.
# The min(., n-1) guard prevents an error if the dataset has fewer than 11 rows.
n_neighbors = min(10, len(rq2_feature_df) - 1)

# WHY two contamination rates [0.03, 0.05]: these bracket the typical 1–5% anomaly
# range in environmental health data. Comparing both tests whether the flagged outliers
# are robust to the assumed contamination level or are artifacts of one specific setting.
for contamination in [0.03, 0.05]:
    # WHY n_estimators=100: the default for IsolationForest and a good balance between
    # variance reduction and runtime. With only ~50-100 rows, 100 trees is more than
    # sufficient for stable anomaly scores.
    iso = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=42,
    )
    iso_labels = iso.fit_predict(rq2_matrix)
    iso_scores = iso.score_samples(rq2_matrix)

    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
    )
    lof_labels = lof.fit_predict(rq2_matrix)
    lof_scores = lof.negative_outlier_factor_

    detail = analysis_model_df[ID_COLS].copy()
    detail["iso_score"] = iso_scores
    detail["iso_outlier"] = iso_labels == -1
    detail["lof_score"] = lof_scores
    detail["lof_outlier"] = lof_labels == -1
    # WHY consensus (AND) of both methods: Isolation Forest detects global anomalies
    # (points that are easy to isolate via random splits) while LOF detects local
    # density anomalies. Requiring agreement from both reduces false positives — a
    # CBSA-year flagged by only one method may be an artifact of that method's
    # assumptions, but agreement across structurally different algorithms is stronger
    # evidence of a genuine anomaly.
    detail["consensus_outlier"] = detail["iso_outlier"] & detail["lof_outlier"]
    detail = detail.sort_values("iso_score")

    rq2_summary_rows.append(
        {
            "contamination": contamination,
            "iforest_outliers": int(detail["iso_outlier"].sum()),
            "lof_outliers": int(detail["lof_outlier"].sum()),
            "consensus_outliers": int(detail["consensus_outlier"].sum()),
        }
    )
    rq2_detail_tables[contamination] = detail.head(10)

rq2_summary = pd.DataFrame(rq2_summary_rows)
print("RQ2 anomaly-detection summary:")
print(f"- Modeling matrix shape: {rq2_feature_df.shape}")
print(f"- LOF n_neighbors: {n_neighbors}")
display(rq2_summary)

for contamination, detail in rq2_detail_tables.items():
    print(f"\nTop Isolation Forest anomaly candidates at contamination={contamination}:")
    display(detail)

In [ ]:
# Initial RQ3 dry run: constrained PrefixSpan feasibility demo.

# WHY only ozone and temperature_2m: PrefixSpan mines sequential patterns, so the
# vocabulary size directly controls the search space. With the full feature set, the
# number of possible subsequences would explode combinatorially. These two features
# are chosen because (a) they showed the most temporal variance in the EDA, making
# year-over-year transitions meaningful, and (b) they represent distinct environmental
# mechanisms (air quality vs. climate), keeping the demo interpretable.
sequence_env_features = ["ozone", "temperature_2m"]
sequence_work_df = analysis_df[ID_COLS + sequence_env_features + DISEASE_COLUMNS].copy()
sequence_bins = {}
for col in sequence_env_features:
    # WHY tercile (3-bin) discretization instead of quartiles: with only ~7 years per
    # CBSA, sequences are short. Fewer bins (3 vs. 4) reduce the token vocabulary,
    # increasing the chance that repeated subsequences reach the minimum support
    # threshold. Terciles still capture low/mid/high gradients.
    sequence_bins[col] = pd.qcut(
        sequence_work_df[col].rank(method="first"),
        3,
        labels=["LOW", "MID", "HIGH"],
    ).astype(str)

# WHY 75th-percentile disease threshold (same as RQ1): maintains consistency across
# research questions — "elevated" prevalence means the same thing in both the
# association-rule and sequential-pattern analyses, enabling direct comparison.
disease_thresholds = sequence_work_df[DISEASE_COLUMNS].quantile(0.75)
sequences = []
sequence_index = []
sequence_lengths = []

for cbsa, group in sequence_work_df.sort_values(ID_COLS).groupby("CBSAFP"):
    seq = []
    ordered_group = group.sort_values("year")
    for row_idx, (_, row) in enumerate(ordered_group.iterrows()):
        seq.append(f"OZONE_{sequence_bins['ozone'].loc[row.name]}")
        seq.append(f"TEMP_{sequence_bins['temperature_2m'].loc[row.name]}")

        high_dx = [
            code
            for code in DISEASE_COLUMNS
            if pd.notna(row[code]) and row[code] >= disease_thresholds[code]
        ]
        # WHY keep only top-2 diseases per year: limits sequence length to prevent
        # PrefixSpan from being overwhelmed by long, noisy sequences. Keeping the two
        # highest-prevalence diseases focuses the analysis on the dominant health
        # burdens in each CBSA-year rather than diluting signal with marginal codes.
        high_dx = sorted(high_dx, key=lambda code: row[code], reverse=True)[:2]
        seq.extend([f"DX:{code}" for code in high_dx])
    sequences.append(seq)
    sequence_index.append(cbsa)
    sequence_lengths.append(len(seq))

prefixspan_runner = PrefixSpan(sequences)
# WHY minlen=2, maxlen=3: a pattern of length 1 is trivial (just a single token).
# Length 2–3 captures meaningful transitions (e.g., "high ozone → respiratory disease")
# without overfitting to the short ~7-year sequences where longer patterns would
# appear in only one or two CBSAs.
prefixspan_runner.minlen = 2
prefixspan_runner.maxlen = 3
# WHY frequent(3) — minimum support of 3 sequences: with only ~10 CBSAs, requiring
# a pattern to appear in at least 3 sequences means it occurs in roughly 30% of
# regions, striking a balance between statistical support and sensitivity.
prefix_patterns = prefixspan_runner.frequent(3)
# WHY filter to mixed env+disease patterns: purely environmental patterns (e.g.,
# "high ozone → high temp") are not actionable for the health research question;
# purely disease patterns ignore the environmental angle. Only mixed patterns
# directly address the RQ of environment-disease temporal co-occurrence.
mixed_patterns = [
    (support, pattern)
    for support, pattern in prefix_patterns
    if any(token.startswith("DX:") for token in pattern)
    and any(not token.startswith("DX:") for token in pattern)
]
mixed_patterns = sorted(mixed_patterns, key=lambda item: (-item[0], len(item[1]), tuple(item[1])))

sequence_summary = pd.DataFrame(
    {
        "CBSAFP": sequence_index,
        "sequence_length": sequence_lengths,
    }
)

print("RQ3 sequence audit:")
print(f"- Number of yearly CBSA sequences: {len(sequences)}")
print(f"- Sequence length range after tokenization: {min(sequence_lengths)} to {max(sequence_lengths)}")
display(sequence_summary.describe())

if mixed_patterns:
    print("\nTop mixed environment-disease PrefixSpan patterns (support >= 3, maxlen <= 3):")
    display(
        pd.DataFrame(
            [
                {"support": support, "pattern": pattern}
                for support, pattern in mixed_patterns[:10]
            ]
        )
    )
else:
    print("\nNo mixed environment-disease PrefixSpan patterns met the support >= 3 and maxlen <= 3 constraints. This negative result is still useful because it shows the aligned yearly data are too sparse for stronger sequence claims at this checkpoint.")

# (F) Works Cited

1. Wang, Yuanlong, et al. "SatHealth: A Multimodal Public Health Dataset with Satellite-based Environmental Factors." Proceedings of the 31st ACM SIGKDD Conference on Knowledge Discovery and Data Mining V. 2. 2025.

2. deleted by me 

3. Han, Jiawei, et al. "Prefixspan: Mining sequential patterns efficiently by prefix-projected pattern growth." proceedings of the 17th international conference on data engineering. Piscataway, NJ, USA: IEEE, 2001.

4. https://github.com/chuanconggao/PrefixSpan-py

5. Usmani, Shan, et al. "Z.“A comparative analysis of apriori and FP-growth algorithms for frequent pattern mining using apache spark”." Proceedings of international scientific research conference. 2021.

6. Cao, Yang, et al. "Anomaly detection based on isolation mechanisms: A survey." Machine Intelligence Research 22.5 (2025): 849-865.

7. Eze, Peter U., et al. "Anomaly detection in endemic disease surveillance data using machine learning techniques." Healthcare. Vol. 11. No. 13. MDPI, 2023.

8. Ma, He, et al. "Discovering sequential patterns and interrelations among multiple diseases in electronic medical records using cSPADE algorithm." Archives of Public Health 83.1 (2025): 100.

9. Chaudhari, Minubhai, and Chirag Mehta. "A Survey on Algorithms for Sequential Pattern Mining." International Journal of Engineering Development and Research 3.4 (2015): 494-496.

10. Hryhoruk, Connor CJ, and Carson K. Leung. "Mining sequential patterns with timelines from digital health data." 2023 IEEE International Conference on Digital Health (ICDH). IEEE, 2023.

11. Bellmann, Louis, et al. "Introducing attribute association graphs to facilitate medical data exploration: development and evaluation using epidemiological study data." JMIR Medical Informatics 12 (2024): e49865.